#Junyang You

# Take home exercise 4

In [0]:
pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
race = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
result = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

In [0]:
df = pit.join(race, on="raceId", how="left") \
        .join(
            result.select("raceId", "driverId", "grid", "positionOrder", "points"),
            on=["raceId", "driverId"],
            how="left"
        )

model_df = df.select(
    "milliseconds",
    "stop",
    "lap",
    "year",
    "round",
    "grid",
    "positionOrder",
    "points"
).dropna()

pdf = model_df.toPandas()

X = pdf.drop(columns=["milliseconds"])
y = pdf["milliseconds"]

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
import mlflow
import mlflow.sklearn

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{user}/HW4_F1")

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import tempfile
import pandas as pd

with mlflow.start_run(run_name="RF_HW4"):

    rf = RandomForestRegressor(
        n_estimators=120,
        max_depth=9,
        random_state=42
    )

    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    mlflow.sklearn.log_model(rf, "rf_model")

    mse = mean_squared_error(y_test, rf_pred)
    mae = mean_absolute_error(y_test, rf_pred)
    r2 = r2_score(y_test, rf_pred)
    rmse = mse ** 0.5

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)

    # artifact 1
    plt.scatter(rf_pred, y_test)
    plt.axhline(0)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    plt.savefig(tmp.name)
    mlflow.log_artifact(tmp.name)
    plt.close()

    # artifact 2
    tmp2 = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
    pd.DataFrame({"pred": rf_pred}).to_csv(tmp2.name, index=False)
    mlflow.log_artifact(tmp2.name)

In [0]:
rf_df = pd.DataFrame({
    "actual": y_test,
    "predicted": rf_pred
})

spark.createDataFrame(rf_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.default.rf_predictions_hw4")

In [0]:
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="Linear_HW4"):

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)

    mlflow.sklearn.log_model(lr, "lr_model")

    mse = mean_squared_error(y_test, lr_pred)
    mae = mean_absolute_error(y_test, lr_pred)
    r2 = r2_score(y_test, lr_pred)
    rmse = mse ** 0.5

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("rmse", rmse)

    # artifact 1
    plt.scatter(lr_pred, y_test)
    plt.axhline(0)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    plt.savefig(tmp.name)
    mlflow.log_artifact(tmp.name)
    plt.close()

    # artifact 2
    tmp2 = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
    pd.DataFrame({"pred": lr_pred}).to_csv(tmp2.name, index=False)
    mlflow.log_artifact(tmp2.name)

In [0]:
lr_df = pd.DataFrame({
    "actual": y_test,
    "predicted": lr_pred
})

spark.createDataFrame(lr_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.default.linear_predictions_hw4")

In [0]:
spark.sql("SELECT * FROM workspace.default.rf_predictions_hw4 LIMIT 10").show()
spark.sql("SELECT * FROM workspace.default.linear_predictions_hw4 LIMIT 10").show()